# GRU Fall Detection — Training

**Run this on a PC with GPU.** The trained `.pth` is then copied to the
Nano and used by `pipeline.ipynb` for real-time inference.

This notebook does *not* depend on TensorRT — only PyTorch + pandas + numpy.

## Input data: CSV format

Each row = one frame from one video. Required columns:

| Column | Type | Description |
|---|---|---|
| `video_id` | str/int | identifies which video; sliding window won't cross videos |
| `frame_idx` | int | frame number within the video (used for ordering) |
| `label` | int | `0` = normal, `1` = fall (constant per video) |
| `x0, y0, s0` | float | keypoint 0: x, y in pixels, score in [0,1] |
| `x1, y1, s1` | float | keypoint 1 |
| ... | ... | ... |
| `x16, y16, s16` | float | keypoint 16 |

Total: **3 metadata columns + 51 keypoint columns = 54 columns**.

Example:
```
video_id,frame_idx,label,x0,y0,s0,x1,y1,s1,...,x16,y16,s16
fall_001,0,1,320.5,100.2,0.95,310.1,90.8,0.93,...
fall_001,1,1,321.0,102.5,0.94,...
fall_001,2,1,...
...
normal_001,0,0,...
```

Keypoint order is COCO-17 (same as MoveNet output): nose, left_eye, right_eye,
left_ear, right_ear, left_shoulder, right_shoulder, left_elbow, right_elbow,
left_wrist, right_wrist, left_hip, right_hip, left_knee, right_knee,
left_ankle, right_ankle.

## Pipeline overview

```
CSV (raw keypoints)
      |
      v
 load_csv_to_videos        ->  {video_id: ((T,17,3), label)}
      |
      v
 KeypointFeatureExtractor  ->  per video (T, 51)
      |
      v
 sliding window            ->  many (30, 51) samples
      |
      v
 FallDetectionGRU
      |
      v
 fall_gru.pth
```

## Steps

| Step | What |
|---|---|
| 0 | Environment check |
| 1 | Sanity check the CSV |
| 2 | Build the dataset (sliding window) |
| 3 | Train GRU |
| 4 | Evaluate on validation set |
| 5 | (Optional) Test on a single video |

## Step 0: Environment check

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import torch

print("Python : {}".format(sys.version.split()[0]))
print("numpy  : {}".format(np.__version__))
print("pandas : {}".format(pd.__version__))
print("torch  : {}".format(torch.__version__))
if torch.cuda.is_available():
    print("GPU    : {}".format(torch.cuda.get_device_name(0)))
else:
    print("GPU    : not available (will use CPU; training will be slow)")

---
## Step 1: Sanity check the CSV

Quick look at the data before we feed it to the dataset class.

In [ ]:
# ============== Config ==============
CSV_PATH = "data/fall_keypoints.csv"     # <-- point at your CSV
# ====================================

if not os.path.exists(CSV_PATH):
    print("[ERROR] CSV not found: {}".format(CSV_PATH))
else:
    df = pd.read_csv(CSV_PATH)
    print("Shape     : {}".format(df.shape))
    print("Columns   : {} ...".format(list(df.columns)[:8]))
    print("Videos    : {}".format(df['video_id'].nunique()))
    print("Frames/vid: min={}, mean={:.1f}, max={}".format(
        df.groupby('video_id').size().min(),
        df.groupby('video_id').size().mean(),
        df.groupby('video_id').size().max()))
    print("\nLabel distribution (per frame):")
    print(df['label'].value_counts().to_string())
    print("\nLabel distribution (per video):")
    per_video = df.groupby('video_id')['label'].first()
    print(per_video.value_counts().to_string())
    print("\nFirst 3 rows:")
    print(df.head(3).to_string())

---
## Step 2: Build the dataset

`FallKeypointCSVDataset` does three things:

1. Calls `load_csv_to_videos(...)` -> a dict `{video_id: ((T,17,3), label)}`,
   sorted by `frame_idx`, with column/label validation.
2. Runs `KeypointFeatureExtractor.extract_sequence(...)` on each video to
   produce `(T, 51)` features. The extractor is *stateful* (speed depends
   on the previous frame), so this MUST be done per video.
3. Slices each `(T, 51)` into overlapping windows of size `seq_len`,
   stepping by `stride`. Windows never cross video boundaries.

Tune `seq_len` and `stride` here:
- `seq_len=30` ≈ 1 s of context at 30 fps. Falls happen in ~1-2 s.
- `stride=5` gives ~6× more samples than non-overlapping; smaller stride
  = more samples but more correlated samples in the same minibatch.

In [ ]:
from fall_model import FallKeypointCSVDataset
from torch.utils.data import DataLoader, random_split

# ============== Dataset config ==============
SEQ_LEN  = 30
STRIDE   = 5
VAL_RATIO = 0.2
BATCH_SIZE = 16
SEED = 42
# ============================================

dataset = FallKeypointCSVDataset(
    csv_path=CSV_PATH,
    seq_len=SEQ_LEN,
    stride=STRIDE,
)
if len(dataset) == 0:
    raise RuntimeError("Dataset is empty — check CSV format and seq_len")

print("\nClass distribution: {}".format(dataset.class_distribution()))

# Quick look at one sample
seq, label = dataset[0]
print("Sample shape: {}, dtype: {}".format(seq.shape, seq.dtype))
print("  feature range: [{:.3f}, {:.3f}]".format(seq.min().item(), seq.max().item()))

# Train/val split
val_n = max(1, int(len(dataset) * VAL_RATIO))
train_n = len(dataset) - val_n
train_set, val_set = random_split(
    dataset, [train_n, val_n],
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)
print("train: {}, val: {}".format(train_n, val_n))

**Note on splitting**: this notebook does a sample-level random split, which
is simple but can leak: windows from the same video may end up in both
train and val. For a stricter eval, do a video-level split (e.g. hold out
20% of `video_id`s before building the dataset). For the typical small
fall dataset (~50-200 videos) the sample-level split is usually OK to
iterate quickly; switch to video-level once you're past prototyping.

---
## Step 3: Train GRU

In [ ]:
import torch.nn as nn
from features import FEATURE_DIM
from fall_model import FallDetectionGRU

# ============== Training config ==============
GRU_SAVE   = "models/fall_gru.pth"
HIDDEN_DIM = 64
NUM_LAYERS = 2
EPOCHS     = 30
LR         = 1e-3
WD         = 1e-4
GRAD_CLIP  = 1.0
# =============================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FallDetectionGRU(
    input_dim=FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
).to(device)
print("Device     : {}".format(device))
print("Params     : {:,}".format(model.num_parameters()))
print("Input dim  : {}".format(FEATURE_DIM))

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- train ---
    model.train()
    total_loss, train_correct, train_total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        logits = model(bx)
        loss = criterion(logits, by)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        train_correct += (logits.argmax(1) == by).sum().item()
        train_total += bx.size(0)
    scheduler.step()

    # --- val ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            val_correct += (model(bx).argmax(1) == by).sum().item()
            val_total += bx.size(0)

    train_loss = total_loss / max(train_total, 1)
    train_acc  = train_correct / max(train_total, 1)
    val_acc    = val_correct / max(val_total, 1)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print("Epoch {:2d}/{}  loss={:.4f}  train_acc={:.4f}  val_acc={:.4f}".format(
            epoch + 1, EPOCHS, train_loss, train_acc, val_acc))

    if val_acc >= best_acc:
        best_acc = val_acc
        os.makedirs(os.path.dirname(GRU_SAVE), exist_ok=True)
        model.save(GRU_SAVE, extra={
            'seq_len': SEQ_LEN,
            'val_acc': val_acc,
            'epoch': epoch + 1,
            'train_acc': train_acc,
        })

print("\n[DONE] best val_acc = {:.4f}".format(best_acc))
print("       saved: {}".format(GRU_SAVE))

In [ ]:
# Plot training curves
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history['train_loss'])
    axes[0].set_title('Train loss'); axes[0].set_xlabel('epoch')
    axes[1].plot(history['train_acc'], label='train')
    axes[1].plot(history['val_acc'],   label='val')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch')
    axes[1].legend(); axes[1].set_ylim(0, 1.01)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed, skipping plot')

---
## Step 4: Evaluate on the validation set

Confusion matrix + per-class precision/recall. Accuracy alone can hide
class imbalance (e.g. 90% normal + a model that always predicts normal
scores 90% accuracy but is useless).

In [ ]:
# Reload the best checkpoint
best_model, ckpt = FallDetectionGRU.load_from(GRU_SAVE, map_location=device)
best_model = best_model.to(device).eval()
print("Reloaded best (val_acc={:.4f}, epoch={})".format(
    ckpt['val_acc'], ckpt['epoch']))

# Confusion matrix
TP = TN = FP = FN = 0
with torch.no_grad():
    for bx, by in val_loader:
        bx = bx.to(device)
        pred = best_model(bx).argmax(1).cpu().numpy()
        true = by.numpy()
        for p, t in zip(pred, true):
            if t == 1 and p == 1: TP += 1
            elif t == 0 and p == 0: TN += 1
            elif t == 0 and p == 1: FP += 1
            else: FN += 1

total = TP + TN + FP + FN
acc       = (TP + TN) / max(total, 1)
precision = TP / max(TP + FP, 1)
recall    = TP / max(TP + FN, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-9)

print("\nConfusion matrix on validation set:")
print("             pred=normal  pred=fall")
print("true=normal     {:5d}      {:5d}".format(TN, FP))
print("true=fall       {:5d}      {:5d}".format(FN, TP))
print("\nMetrics:")
print("  accuracy : {:.4f}".format(acc))
print("  precision: {:.4f}  (when we say 'fall', how often are we right)".format(precision))
print("  recall   : {:.4f}  (of actual falls, how many do we catch)".format(recall))
print("  F1       : {:.4f}".format(f1))

Interpreting these for fall detection:
- **Recall is critical** — missing a real fall (FN) is worse than a false alarm.
- **Precision matters too** — too many false alarms make the system useless.
- A typical good first model: recall ≥ 0.9, precision ≥ 0.7. Tune the
  decision threshold (and the `FALL_THRESHOLD` / `ALARM_FRAMES` in
  `pipeline.ipynb`) to balance the two.

---
## Step 5 (optional): Test on one video

Pick a single `video_id`, slide the model over it, and plot fall
probability vs frame. Useful for visualizing what the model thinks.

In [ ]:
from fall_model import load_csv_to_videos
from features import KeypointFeatureExtractor
from collections import deque

# ============== Test config ==============
TEST_VIDEO_ID = None     # None = pick the first fall video automatically
# =========================================

videos = load_csv_to_videos(CSV_PATH, verbose=False)

if TEST_VIDEO_ID is None:
    # Pick the first fall video
    for vid, (kp, lbl) in videos.items():
        if lbl == 1:
            TEST_VIDEO_ID = vid
            break

if TEST_VIDEO_ID is None or TEST_VIDEO_ID not in videos:
    print("No suitable test video found")
else:
    kpts_seq, label = videos[TEST_VIDEO_ID]
    print("Testing on video: {} (label={}, T={})".format(
        TEST_VIDEO_ID, label, kpts_seq.shape[0]))

    ext = KeypointFeatureExtractor()
    feats = ext.extract_sequence(kpts_seq)    # (T, 51)

    # Slide a window across the video, predict for each window
    probs = []
    buf = deque(maxlen=SEQ_LEN)
    for t in range(feats.shape[0]):
        buf.append(feats[t])
        if len(buf) == SEQ_LEN:
            seq = torch.from_numpy(np.stack(buf)).float().unsqueeze(0).to(device)
            with torch.no_grad():
                p = torch.softmax(best_model(seq), 1)[0, 1].item()
            probs.append(p)
        else:
            probs.append(np.nan)

    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(11, 3))
        plt.plot(probs)
        plt.axhline(0.5, ls='--', color='gray', label='0.5')
        plt.axhline(0.7, ls=':',  color='red',  label='0.7 (alarm threshold)')
        plt.title('Fall probability over time — {} (true label={})'.format(
            TEST_VIDEO_ID, label))
        plt.xlabel('frame'); plt.ylabel('P(fall)')
        plt.ylim(-0.02, 1.02); plt.legend()
        plt.tight_layout(); plt.show()
    except ImportError:
        print('first/last 10 probs:', probs[:10], '...', probs[-10:])

---
## Next steps

1. **Copy the .pth to the Nano**:
   ```bash
   scp models/fall_gru.pth nano@<NANO_IP>:~/movenet_jeston_nano/models/
   ```

2. On the Nano, run `pipeline.ipynb` Step 3 to do real-time inference.

3. If recall is too low, consider:
   - More fall data, especially varied angles/light
   - Class weights in `CrossEntropyLoss(weight=...)` if the dataset is imbalanced
   - Larger `seq_len` so the model sees more pre-fall context

4. If precision is too low (too many false alarms):
   - Add more confusing-action data (sit-down, lie-down, bend-over) labeled as `normal`
   - Raise `FALL_THRESHOLD` in `pipeline.ipynb`